# CrisisOps — GRPO Training Notebook

Train an LLM to detect deceptive software engineers and recover failing projects.

**One-click in Colab.** Run cells top to bottom.

## What this demonstrates
- **Theme 1**: Two-agent interaction — GRPO-trained PM vs LLM-powered deceptive member
- **Theme 2**: Long-horizon planning with memory buffer compression every 8 steps
- **Theme 3.1**: Real tool use — Jira adapter, observable signal queries, counterfactual reward
- **Theme 4**: Adaptive curriculum — crisis generator tracks agent weaknesses via EMA

**Novel mechanisms implemented:**
1. Dynamic candor evolution — caught liars become more honest; unchecked liars grow bolder
2. Social testimony graph — query one member about another; honest vs allied deceptive give different signals
3. Alibi coordination — deceptive alliance members give consistent coordinated alibis
4. Political capital — second earned resource; spend to compel truth or trigger whistleblower
5. LLM-powered adversarial agent — Ollama (qwen2.5:3b) with OpenAI fallback, or rule-based; contextual adaptive lies
6. Long-horizon memory buffer — episode state compressed every N steps and injected into observation

---

**Before you run (read once)**  
- **Runtime → Change runtime type → GPU** (T4 is enough; A100 is faster).  
- In **Cell 2**, set `REPO_URL` / **`REPO_BRANCH`** (`Aryan` for this team). **Exit 128** on `git clone` is usually: (a) old `/content/crisisops` not removed — run `!rm -rf /content/crisisops` then re-run the cell; (b) **private repo** — set `os.environ["GITHUB_TOKEN"]="ghp_..."` before Cell 2. If clone fails, the cell now prints **git stderr** so you can see the exact reason.  
- **Training:** run cells **1 → 2 → 3 → 4** in order. Cell 4 runs GRPO and saves **`crisisops_run/reward_log.json`** (rewards) and training logs; check the Colab output for `Training log:` and `train/loss` style lines from the trainer.

In [ ]:
# Cell 1: Install dependencies — then RESTARTS RUNTIME
import subprocess, sys, os

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

print('Installing HF stack (no Unsloth — using vanilla TRL)...')
pip('torch>=2.1.0')
pip('transformers==4.56.2')
pip('trl==0.19.1')
pip('accelerate>=0.27.0')
pip('peft>=0.10.0')
pip('bitsandbytes>=0.43.0')   # 4-bit quantization without Unsloth
pip('datasets>=3.2.0')
pip('matplotlib', 'scipy', 'openai>=1.0.0', 'pytest')
print('✓ Dependencies installed')
print()
print('Restarting runtime — after restart run Cells 2 → 3 → 4. Do NOT re-run Cell 1.')
os.kill(os.getpid(), 9)

In [ ]:
os.chdir('/content')
# Cell 2: Clone repo and set up environment
import os, sys, subprocess, shutil, time

IN_COLAB = 'google.colab' in str(get_ipython())

# --- Git remote + branch (must exist on GitHub: same names as in the web UI) ---
REPO_URL    = 'https://github.com/vedchamp07/crisisops'
REPO_BRANCH = 'Aryan'

# Private repo on Colab: create a token at github.com/settings/tokens (read-only repo scope),
# then in Colab: os.environ['GITHUB_TOKEN'] = 'ghp_...' before this cell, OR use Colab secrets.
_GH_TOKEN = os.environ.get('GITHUB_TOKEN') or os.environ.get('GH_TOKEN')


def _clone_url(base: str) -> str:
    if not _GH_TOKEN or 'github.com' not in base:
        return base
    # https://github.com/org/repo -> https://<token>@github.com/org/repo
    return base.replace('https://', f'https://{_GH_TOKEN}@', 1)


if IN_COLAB:
    os.environ.setdefault('GIT_TERMINAL_PROMPT', '0')
    repo_path = '/content/crisisops'
    # Exit 128 is often: (1) destination exists and is not empty, (2) private repo without auth.
    if os.path.exists(repo_path):
        subprocess.run(['rm', '-rf', repo_path], check=False)
        shutil.rmtree(repo_path, ignore_errors=True)
        time.sleep(0.1)
    if os.path.exists(repo_path):
        raise RuntimeError(
            f'Could not delete {repo_path}. In a new cell run: !rm -rf {repo_path}  then re-run this cell.'
        )

    url = _clone_url(REPO_URL)
    cmd = ['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, url, repo_path]
    print('git clone ...', REPO_URL, '-b', REPO_BRANCH, '->', repo_path)
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0:
        print('--- git stderr ---\n', p.stderr)
        print('--- git stdout ---\n', p.stdout)
        hint = ''
        if 'repository not found' in (p.stderr or '').lower() or 'authentication failed' in (p.stderr or '').lower():
            hint = (
                ' Repo may be private: set os.environ["GITHUB_TOKEN"] = "ghp_..." before this cell '
                '(token needs repo read access).'
            )
        elif 'already exists' in (p.stderr or '').lower() or 'not empty' in (p.stderr or '').lower():
            hint = f' Run !rm -rf {repo_path} and re-run.'
        raise RuntimeError(f'git clone failed (exit {p.returncode}).{hint}')

    os.chdir(repo_path)
else:
    pass

repo_root = os.path.abspath(os.getcwd())
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print('Working directory:', repo_root)

from env.environment import CrisisOpsEnv
from env.actions import ACTION_COSTS
print(f'✓ Environment loaded — {len(ACTION_COSTS)} actions registered')
assert len(ACTION_COSTS) >= 16, f'Expected 16+ actions, got {len(ACTION_COSTS)}'

from env.state import TeamMember, ProjectState
required_tm = [
    'alliance_id', 'times_cross_verified', 'caught_this_episode',
    'is_llm_agent', 'pm_actions_toward_member', 'prior_statements',
]
tm_fields = set(TeamMember.__dataclass_fields__.keys())
missing_tm = [f for f in required_tm if f not in tm_fields]
if missing_tm:
    raise AssertionError(
        f'Missing TeamMember field(s) {missing_tm} — your clone is behind GitHub. '
        f'Push/merge the latest `env/state.py` from your machine, or set REPO_BRANCH to a branch that has it, '
        f'then re-run this cell. (Current REPO_URL={REPO_URL!r}, REPO_BRANCH={REPO_BRANCH!r})'
    )
print('✓ All TeamMember fields present')

required_ps = ['political_capital', 'memory_buffer', 'memory_compression_interval']
ps_fields = set(ProjectState.__dataclass_fields__.keys())
missing_ps = [f for f in required_ps if f not in ps_fields]
if missing_ps:
    raise AssertionError(f'Missing ProjectState field(s) {missing_ps}')

print('✓ All ProjectState fields present')

from scenarios.level1 import get_random_level1_scenario
from reward.counterfactual import counterfactual_reward
env = CrisisOpsEnv(scenario_fn=get_random_level1_scenario(), curriculum_level=1)
obs = env.reset(seed=42)
assert 'political_capital' in obs
obs2, r, done, info = env.step({'action_type': 'query_status', 'params': {}})
print('✓ Smoke test passed — PC=%s, budget=%s' % (obs2.get('political_capital'), obs2.get('budget_remaining')))

In [ ]:
# Cell 3: Calibration — sanity check before training
from calibration.calibrate import run_calibration

print('Running calibration (20 episodes)...')
result = run_calibration(n_episodes=20, seed=99)
print(f"Greedy mean: {result['greedy_mean']:.3f}")
print(f"Oracle mean: {result['oracle_mean']:.3f}")
print(f"Gap:         {result['gap']:.3f}")
print(f"Status:      {result['status']}")
print()

greedy = result['greedy_mean']
oracle = result['oracle_mean']
gap    = result['gap']

# Hard failure only: env is broken (oracle ≤ greedy, or gap essentially zero)
if oracle <= greedy:
    raise RuntimeError(f'Oracle ({oracle:.3f}) ≤ Greedy ({greedy:.3f}) — env reward is inverted or broken.')
if gap < 0.05:
    raise RuntimeError(f'Gap too small ({gap:.3f}) — greedy and oracle perform identically. Check env.')

# Soft warnings — these are calibration range targets, not blockers
if greedy > 0.60:
    print(f'[WARN] Greedy {greedy:.3f} slightly above target [0.45–0.55] — training still has room to improve.')
if gap < 0.20:
    print(f'[WARN] Gap {gap:.3f} slightly below target [0.20–0.35] — signal is thinner but sufficient.')

print('✓ Calibration OK — environment is healthy, proceeding to training.')

In [ ]:
# Cell 4: GRPO Training (vanilla TRL + bitsandbytes)
# =============================================================================
# HOW TO RUN
#   1. Set NUM_EPISODES below (50 = quick test, 300 = full run).
#   2. Optional: OPENAI_API_KEY in Colab Secrets for LLM deceptive agent.
#   3. Run this cell — training time depends on GPU (~45 min/100 eps on T4).
# =============================================================================
import os, sys, shutil, re as _re

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# --- Section 1: Patch grpo_trainer.py for Colab (vanilla stack, no Unsloth) ---
shutil.rmtree("/content/crisisops/training/__pycache__", ignore_errors=True)
for m in list(sys.modules.keys()):
    if "grpo_trainer" in m:
        del sys.modules[m]

_trainer_path = "/content/crisisops/training/grpo_trainer.py"
with open(_trainer_path) as f:
    _src = f.read()

if "VANILLA_PATCHED" not in _src:
    _OLD_START = "    try:\n        from unsloth import FastLanguageModel"
    _OLD_END = "    os.makedirs(output_dir, exist_ok=True)"
    assert _OLD_START in _src, "Unsloth block not found — wrong grpo_trainer version?"
    _si = _src.index(_OLD_START)
    _ei = _src.index(_OLD_END, _si) + len(_OLD_END)
    _NEW_BLOCK = """    # VANILLA_PATCHED
    from trl import GRPOTrainer, GRPOConfig
    from transformers import (AutoModelForCausalLM, AutoTokenizer,
                               BitsAndBytesConfig, TrainerCallback)
    from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
    import torch

    random.seed(seed)
    generator = CrisisGenerator(curriculum_level=curriculum_level)
    curriculum = CurriculumManager(starting_level=curriculum_level)
    current_level = curriculum_level

    _bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=_bnb, device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = model.config.eos_token_id
    model.config.use_cache = False
    if hasattr(model, "generation_config") and model.generation_config is not None:
        model.generation_config.use_cache = False

    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, LoraConfig(
        r=LORA_RANK, lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODULES,
        lora_dropout=LORA_DROPOUT, bias="none",
        task_type=TaskType.CAUSAL_LM,
    ))
    model.print_trainable_parameters()

    os.makedirs(output_dir, exist_ok=True)"""
    _src = _src[:_si] + _NEW_BLOCK + _src[_ei:]
    _src = _re.sub(
        r"\n    from env\.crisis_generator import CrisisGenerator\n"
        r"    from training\.curriculum import CurriculumManager\n"
        r"\n    random\.seed\(seed\)\n"
        r"    generator = CrisisGenerator[^\n]*\n"
        r"    curriculum = CurriculumManager[^\n]*\n"
        r"    current_level = curriculum_level\n",
        "\n", _src, count=1,
    )

_s_old = "trainer = GRPOTrainer(\n        model=model,\n        tokenizer=tokenizer,"
_s_new = "trainer = GRPOTrainer(\n        model=model,\n        processing_class=tokenizer,"
if _s_old in _src:
    _src = _src.replace(_s_old, _s_new)
_src = _src.replace('"train/reward"', '"rewards"')

with open(_trainer_path, "w") as f:
    f.write(_src)

shutil.rmtree("/content/crisisops/training/__pycache__", ignore_errors=True)
for m in list(sys.modules.keys()):
    if "grpo_trainer" in m:
        del sys.modules[m]

# --- Section 2: Runtime injections (match T4 14.5GB) ---
import training.grpo_trainer as _gt
from env.crisis_generator import CrisisGenerator
from training.curriculum import CurriculumManager

_gt.CrisisGenerator = CrisisGenerator
_gt.CurriculumManager = CurriculumManager
_gt.GRPO_NUM_GENERATIONS = 4
_gt.GRPO_BATCH_SIZE = 4

# --- Section 3: Train ---
NUM_EPISODES = 300  # 50 for a quick test
OUTPUT_DIR = "./crisisops_run"
SEED = 42
START_LEVEL = 1
# os.environ["OPENAI_API_KEY"] = ""  # optional — Colab Secrets recommended

from training.grpo_trainer import train

train(
    curriculum_level=START_LEVEL,
    num_episodes=NUM_EPISODES,
    output_dir=OUTPUT_DIR,
    seed=SEED,
)
print("Training complete")


In [ ]:
# Cell 5: Training curve visualization
import json, os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import gaussian_filter1d

OUTPUT_DIR = './crisisops_run'
log_path = os.path.join(OUTPUT_DIR, 'reward_log.json')

if not os.path.exists(log_path):
    raise FileNotFoundError(f'reward_log.json not found at {log_path}. Run Cell 4 first.')

with open(log_path) as f:
    log = json.load(f)

episodes = [r['episode'] for r in log]
rewards  = [r['reward']  for r in log]
levels   = [r['level']   for r in log]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Reward curve with curriculum shading ---
import numpy as np
ax = axes[0]
n0 = len(rewards)
sg = min(3, max(1, n0 // 10)) if n0 else 1
smoothed = gaussian_filter1d(rewards, sigma=sg) if n0 > 1 else np.asarray(rewards, dtype=float)
ax.plot(episodes, rewards, alpha=0.25, color='#2563eb', linewidth=0.8, label='Raw CF reward')
ax.plot(episodes, smoothed, color='#1d4ed8', linewidth=2.0, label='Smoothed')
ax.axhline(0, color='#dc2626', linestyle='--', linewidth=1.0, label='Greedy baseline (0)')

# Shade curriculum level regions
level_colors = {1: '#eff6ff', 2: '#f0fdf4', 3: '#fefce8', 4: '#fff1f2'}
level_names  = {1: 'L1', 2: 'L2', 3: 'L3', 4: 'L4'}
prev_ep, prev_lv = (episodes[0] if episodes else 0), (levels[0] if levels else 1)
for ep, lv in zip(episodes, levels):
    if lv != prev_lv:
        ax.axvspan(prev_ep, ep, alpha=0.4, color=level_colors.get(prev_lv, '#f5f5f5'))
        ax.text((prev_ep + ep) / 2, ax.get_ylim()[0] + 0.02,
                level_names[prev_lv], ha='center', fontsize=8, color='#6b7280')
        prev_ep, prev_lv = ep, lv
ax.axvspan(prev_ep, episodes[-1] if episodes else 0,
           alpha=0.4, color=level_colors.get(prev_lv, '#f5f5f5'))

ax.set_xlabel('Training episode', fontsize=11)
ax.set_ylabel('Counterfactual reward (agent − greedy PM)', fontsize=11)
ax.set_title('CrisisOps — GRPO Training Curve', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- Plot 2: Reward distribution (early vs late training) ---
ax2 = axes[1]
n = len(rewards)
first_quarter = rewards[:max(1, n//4)]
last_quarter  = rewards[3*n//4:] if n else []
if not last_quarter and n:
    last_quarter = rewards[-max(1, n//4):]
ax2.hist(first_quarter, bins=20, alpha=0.6, color='#dc2626', label=f'First 25% of training (mean={np.mean(first_quarter):.3f})')
ax2.hist(last_quarter,  bins=20, alpha=0.6, color='#2563eb', label=f'Last 25% of training (mean={np.mean(last_quarter):.3f})')
ax2.axvline(0, color='black', linestyle='--', linewidth=1.0, label='Greedy baseline')
ax2.set_xlabel('Counterfactual reward', fontsize=11)
ax2.set_ylabel('Episode count', fontsize=11)
ax2.set_title('Reward Distribution: Early vs Late Training', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curve_final.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Episodes trained: {len(episodes)}')
print(f'Mean reward (all): {sum(rewards)/len(rewards):.4f}')
print(f'Mean reward (last 25%): {sum(last_quarter)/len(last_quarter):.4f}')
print(f'Curriculum levels reached: {sorted(set(levels))}')
print(f'Plot saved to: {OUTPUT_DIR}/training_curve_final.png')

In [ ]:
# Cell 6: Qualitative evaluation — trained vs untrained
import json, os
from env.environment import CrisisOpsEnv
from scenarios.level3 import scenario_adversarial_majority
from reward.baseline import GreedyPMBaseline

OUTPUT_DIR = './crisisops_run'
EVAL_SEED  = 999

def run_greedy_episode(seed):
    env = CrisisOpsEnv(scenario_fn=scenario_adversarial_majority, curriculum_level=3)
    obs = env.reset(seed=seed)
    agent = GreedyPMBaseline()
    done, actions = False, []
    while not done:
        action = agent.act(env._state)
        obs, reward, done, info = env.step(action)
        actions.append(action.get('action_type', '?'))
    return reward, actions, info

print('=== GREEDY PM BASELINE ===')
g_reward, g_actions, g_info = run_greedy_episode(EVAL_SEED)
print(f'Reward:           {g_reward:.4f}')
print(f'Actions:          {g_actions}')
print(f'Cross-verify:     {g_info["cross_verify_rate"]:.2f}')
print(f'Active crises:    {g_info["active_crises"]}')
print()

checkpoint_path = os.path.join(OUTPUT_DIR, 'checkpoint_ep50')
if not os.path.exists(checkpoint_path):
    print(f'No checkpoint at {checkpoint_path} — run Cell 4 first.')
else:
    print('=== TRAINED AGENT (checkpoint_ep50) ===')
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        from peft import PeftModel
        from training.grpo_trainer import format_observation_as_prompt, parse_action_from_response, SYSTEM_PROMPT
        import torch

        _bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        # Load base model then apply saved LoRA adapter
        from training.grpo_trainer import MODEL_NAME
        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, quantization_config=_bnb, device_map="auto"
        )
        model = PeftModel.from_pretrained(base_model, checkpoint_path)
        model.eval()
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        env = CrisisOpsEnv(scenario_fn=scenario_adversarial_majority, curriculum_level=3)
        obs = env.reset(seed=EVAL_SEED)
        done, actions, step, reward = False, [], 0, 0.0

        while not done and step < 30:
            messages = [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': format_observation_as_prompt(obs)},
            ]
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = tokenizer(text, return_tensors='pt').to(model.device)
            with torch.no_grad():
                out = model.generate(
                    **inputs, max_new_tokens=256,
                    temperature=0.1, do_sample=True,
                    pad_token_id=tokenizer.eos_token_id,
                )
            response = tokenizer.decode(
                out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
            )
            action = parse_action_from_response(response)
            obs, reward, done, info = env.step(action)
            actions.append(action.get('action_type', '?'))
            step += 1

        print(f'Reward:           {reward:.4f}')
        print(f'Actions:          {actions}')
        print(f'Cross-verify:     {info["cross_verify_rate"]:.2f}')
        print(f'Active crises:    {info["active_crises"]}')
        print(f'vs greedy:        {reward - g_reward:+.4f}')

    except Exception as e:
        print(f'Error loading trained model: {e}')

In [ ]:
# Cell 7: Jira/Linear adapter demo
# Shows real tool-use capability (dry_run=True by default)
from deployment.jira_adapter import JiraAdapter

adapter = JiraAdapter(dry_run=True)

# Simulate what the agent would write to Jira after a successful recovery
recovery_plan = {
    'plan_summary': (
        'CrisisOps agent detected 2 deceptive engineers (alibi coordination identified). '
        'Reassigned 3 critical-path tasks. Used force_truth on dev_am1 (actual=0.08 vs reported=0.72). '
        'Triggered whistleblower tip at PC=8 to identify worst liar. '
        'Client satisfaction maintained at 7.8. All 3 crises resolved.'
    ),
    'risk_items': [
        'dev_am1 and dev_am2 formed alibi alliance — monitoring continues',
        'Schema drift event at step 8 acknowledged within window',
        'Budget consumed 18/20 — future episodes need earlier submission',
    ],
    'timeline': '2026-05-15',
}

adapter.submit_recovery_plan(recovery_plan)
print('✓ Recovery plan submitted to Jira (dry_run=True — set dry_run=False with real credentials to write live)')